In [0]:
# Imports
import requests
import time
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType
)
from pyspark.sql.functions import lit, current_timestamp

print("Imports successful")

In [0]:
# Define our countries with coordinates
# Open-Meteo works on lat/lon, not country codes
# We use capital cities as the representative point per country

COUNTRIES = [
    {"country_code": "KE", "country_name": "Kenya",        "lat": -1.286,  "lon": 36.817},
    {"country_code": "UG", "country_name": "Uganda",       "lat":  0.347,  "lon": 32.583},
    {"country_code": "TZ", "country_name": "Tanzania",     "lat": -6.173,  "lon": 35.739},
    {"country_code": "ET", "country_name": "Ethiopia",     "lat":  9.025,  "lon": 38.747},
    {"country_code": "GH", "country_name": "Ghana",        "lat":  5.603,  "lon": -0.187},
    {"country_code": "NG", "country_name": "Nigeria",      "lat":  9.058,  "lon":  7.499},
    {"country_code": "ZA", "country_name": "South Africa", "lat": -25.746, "lon": 28.188},
    {"country_code": "SN", "country_name": "Senegal",      "lat": 14.693,  "lon": -17.447},
    {"country_code": "ML", "country_name": "Mali",         "lat": 12.650,  "lon": -8.000},
    {"country_code": "MZ", "country_name": "Mozambique",   "lat": -25.966, "lon": 32.573},
    {"country_code": "RW", "country_name": "Rwanda",       "lat": -1.943,  "lon": 29.874},
    {"country_code": "ZM", "country_name": "Zambia",       "lat": -15.417, "lon": 28.283},
    {"country_code": "ZW", "country_name": "Zimbabwe",     "lat": -17.829, "lon": 31.052},
    {"country_code": "CM", "country_name": "Cameroon",     "lat":  3.848,  "lon": 11.502},
    {"country_code": "CI", "country_name": "Cote d'Ivoire","lat":  5.354,  "lon": -4.005},
]

print(f"Defined {len(COUNTRIES)} countries with coordinates")

In [0]:
# Cell 3 — the API call function
# Open-Meteo's archive API returns daily data
# We request daily precipitation and temperature, then, 
# aggregate to monthly averages in Silver layer
# For now Bronze = raw daily records

def fetch_climate(country: dict, start: str, end: str) -> list:
    """
    Fetches daily climate data from Open-Meteo archive API
    for a single country (by lat/lon).

    Returns a flat list of daily records.
    """
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude":   country["lat"],
        "longitude":  country["lon"],
        "start_date": start,
        "end_date":   end,
        "daily":      "precipitation_sum,temperature_2m_mean",
        "timezone":   "UTC",
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    # Open-Meteo returns parallel arrays — dates[], precipitation[], temperature[]
    # We zip them together into row-level records
    dates  = data["daily"]["time"]
    precip = data["daily"]["precipitation_sum"]
    temps  = data["daily"]["temperature_2m_mean"]

    records = []
    for date, p, t in zip(dates, precip, temps):
        records.append({
            "country_code": country["country_code"],
            "country_name": country["country_name"],
            "latitude":     country["lat"],
            "longitude":    country["lon"],
            "date":         date,           # "YYYY-MM-DD"
            "year":         int(date[:4]),
            "month":        int(date[5:7]),
            "precipitation_mm":   float(p) if p is not None else None,
            "temperature_c_mean": float(t) if t is not None else None,
        })

    return records

# Test on Kenya first
test_records = fetch_climate(COUNTRIES[0], "2023-01-01", "2023-01-05")
print(f"Sample record: {test_records[0]}")

In [0]:
# Cell 4 — fetch all countries with retry + exponential backoff
# Exponential backoff = if request fails, wait, then try again

import time

def fetch_climate_with_retry(country: dict, start: str, end: str, max_retries: int = 3) -> list:
    """
    Wraps fetch_climate with retry logic.
    If a 429 is received, waits and retries up to max_retries times.
    """
    wait_time = 10  # start with 10 seconds on first failure

    for attempt in range(max_retries + 1):
        try:
            records = fetch_climate(country, start, end)
            return records
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 429 and attempt < max_retries:
                print(f"  Rate limited. Waiting {wait_time}s before retry {attempt + 1}/{max_retries}...")
                time.sleep(wait_time)
                wait_time *= 2  # double the wait each retry: 10 → 20 → 40
            else:
                raise  # if not 429 or out of retries, raise the error

# Only fetch the 6 countries that failed — no need to re-fetch the 9 that succeeded
FAILED_COUNTRIES = [
    {"country_code": "MZ", "country_name": "Mozambique",    "lat": -25.966, "lon": 32.573},
    {"country_code": "RW", "country_name": "Rwanda",        "lat": -1.943,  "lon": 29.874},
    {"country_code": "ZM", "country_name": "Zambia",        "lat": -15.417, "lon": 28.283},
    {"country_code": "ZW", "country_name": "Zimbabwe",      "lat": -17.829, "lon": 31.052},
    {"country_code": "CM", "country_name": "Cameroon",      "lat":  3.848,  "lon": 11.502},
    {"country_code": "CI", "country_name": "Cote d'Ivoire", "lat":  5.354,  "lon": -4.005},
]

retry_records = []

for country in FAILED_COUNTRIES:
    print(f"Fetching: {country['country_name']}...", end=" ")
    try:
        records = fetch_climate_with_retry(country, "2010-01-01", "2023-12-31")
        retry_records.extend(records)
        print(f"{len(records)} daily records")
    except Exception as e:
        print(f"FAILED after retries — {e}")
    time.sleep(8)  # 8 seconds between each country this time

print(f"\nRetry records fetched: {len(retry_records)}")
print(f"Previous records: {len(all_records)}")
print(f"Combined total: {len(all_records) + len(retry_records)}")

In [0]:
# Combine original successful records with retry records

all_records_complete = all_records + retry_records
print(f"Complete dataset: {len(all_records_complete)} records across 15 countries")

# Expected: 15 countries × 5113 days = 76,695 records
expected = 15 * 5113
print(f"Expected: {expected} records")
print(f"Match: {len(all_records_complete) == expected}")

In [0]:
# Cell 5 — define schema and create Spark DataFrame
# Notice precipitation and temperature are nullable (True)
# Real climate data has missing days — we handle nulls properly
# rather than dropping them or filling with zeros

schema = StructType([
    StructField("country_code",       StringType(),  True),
    StructField("country_name",       StringType(),  True),
    StructField("latitude",           DoubleType(),  True),
    StructField("longitude",          DoubleType(),  True),
    StructField("date",               StringType(),  True),
    StructField("year",               IntegerType(), True),
    StructField("month",              IntegerType(), True),
    StructField("precipitation_mm",   DoubleType(),  True),
    StructField("temperature_c_mean", DoubleType(),  True),
])

df_climate = spark.createDataFrame(all_records_complete, schema=schema)

# Add Bronze metadata
df_climate = (df_climate
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source",      lit("open_meteo_api"))
    .withColumn("indicator",   lit("climate_daily"))
)

print(f"DataFrame shape: {df_climate.count()} rows, {len(df_climate.columns)} columns")
df_climate.printSchema()

In [0]:
# Write to Bronze Delta table

BRONZE_PATH = "/Volumes/workspace/default/econmate/bronze/open_meteo_climate"

(df_climate.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(BRONZE_PATH)
)

print(f"Written to Bronze: {BRONZE_PATH}")

In [0]:
# verify

df_check = spark.read.format("delta").load(BRONZE_PATH)

print(f"Total rows in Bronze climate table: {df_check.count()}")
print(f"\nRows per country:")
display(
    df_check.groupBy("country_code", "country_name")
            .count()
            .orderBy("country_code")
)

In [0]:
# Check on the actual values
# We want to see that precipitation and temperature look resonable

print("Climate value ranges per country:")
display(
    df_check.groupBy("country_code")
            .agg(
                {"precipitation_mm":   "avg",
                 "temperature_c_mean": "avg"}
            )
            .orderBy("country_code")
)

In [0]:
# Run this ONCE to create a secret scope and store your ACLED credentials
# After this cell runs successfully, delete it — never leave credentials in notebooks

import subprocess

# Create a secret scope for EconMate
scope_name = "econmate"

# You'll use dbutils.secrets to store values securely
# These are stored encrypted in Databricks — not visible in notebooks or logs

dbutils.secrets.help()  # confirm secrets utility is available